# GitHub Repository Explainer Agent



In [46]:
!pip install -q sentence-transformers faiss-cpu langchain langchain-groq GitPython langgraph


In [47]:
import os
import pickle
import shutil
import getpass
from pathlib import Path

import numpy as np
import pandas as pd

from git import Repo
from sentence_transformers import SentenceTransformer
import faiss

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


In [48]:
REPO_DIR = "cloned_repo"
INDEX_DIR = "faiss_index"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 180
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
TOP_K = 5
SUPPORTED_EXTENSIONS = {
    ".py", ".cpp", ".c", ".h", ".java", ".js", ".ts", ".tsx", ".jsx",
    ".html", ".css", ".md", ".txt", ".json", ".yaml", ".yml",
}
IGNORED_DIRS = {
    ".git", "__pycache__", ".ipynb_checkpoints", "node_modules",
    "dist", "build", "venv", "env", ".venv", "site-packages",
}
IGNORED_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".pdf", ".exe", ".dll",
    ".so", ".bin", ".ico", ".woff", ".woff2", ".ttf", ".eot",
    ".zip", ".tar", ".gz", ".lock",
}
SYSTEM_PROMPT = """You are an expert software engineer explaining a codebase to another engineer.

Rules you MUST follow:
1. Answer ONLY using the provided repository context below. Do not use outside knowledge about the framework/library unless it's needed to explain a general concept the code relies on.
2. If the answer cannot be found in the context, respond exactly with: "I couldn't find enough information inside this repository."
3. Always mention the specific filename(s) your answer is based on.
4. Do not hallucinate function names, file names, or behavior that isn't shown in the context.
5. Be clear and structured. Use bullet points or short paragraphs where helpful.
"""


In [49]:
def clone_repo(repo_url: str, dest_dir: str = REPO_DIR) -> str:
    """
    Clone a public GitHub repository into a local directory.
    Args:
        repo_url: HTTPS URL of the public GitHub repository.
        dest_dir: Local folder to clone into (wiped first if it exists).
    Returns:
        The local path to the cloned repository.
    Raises:
        RuntimeError: if cloning fails (bad URL, private repo, network issue).
    """
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)

    try:
        Repo.clone_from(repo_url, dest_dir, depth=1)
    except Exception as exc:
        raise RuntimeError(f"Failed to clone '{repo_url}': {exc}") from exc

    print(f"Cloned '{repo_url}' into '{dest_dir}'.")
    return dest_dir


In [50]:
def read_repository(repo_path: str) -> list[dict]:
    """
    Walk a cloned repository and read all supported source/doc files.
    Args:
        repo_path: Local path to the cloned repository.
    Returns:
        List of dicts: {"file_path": <relative path>, "content": <file text>}
    """
    files_data = []

    for root, dirnames, filenames in os.walk(repo_path):
        dirnames[:] = [d for d in dirnames if d not in IGNORED_DIRS]
        for filename in filenames:
            ext = Path(filename).suffix.lower()

            if ext in IGNORED_EXTENSIONS or ext not in SUPPORTED_EXTENSIONS:
                continue

            full_path = os.path.join(root, filename)
            rel_path = os.path.relpath(full_path, repo_path)

            try:
                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()
            except (UnicodeDecodeError, OSError):
                continue

            if content.strip():
                files_data.append({"file_path": rel_path, "content": content})

    print(f"Read {len(files_data)} supported files from '{repo_path}'.")
    return files_data


In [51]:
def chunk_code(files_data: list[dict], chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[dict]:
    """
    Split file contents into overlapping, line-aware chunks.

    Rather than blindly slicing every N characters (which can cut a function
    definition in half), this packs whole lines into a chunk until the size
    limit is hit, then starts the next chunk with ~`overlap` characters of
    trailing context so nothing meaningful is lost at chunk boundaries.

    Args:
        files_data: Output of read_repository() -> [{"file_path", "content"}, ...]
        chunk_size: Target max characters per chunk.
        overlap: Approximate character overlap between consecutive chunks.

    Returns:
        List of dicts: {"file_path", "chunk", "chunk_id"}
    """
    all_chunks = []

    for file_entry in files_data:
        file_path = file_entry["file_path"]
        lines = file_entry["content"].splitlines(keepends=True)

        current_lines: list[str] = []
        current_len = 0
        chunk_id = 0

        i = 0
        while i < len(lines):
            line = lines[i]
            current_lines.append(line)
            current_len += len(line)
            i += 1

            is_last_line = i == len(lines)
            if current_len >= chunk_size or is_last_line:
                chunk_text = "".join(current_lines).strip()
                if chunk_text:
                    all_chunks.append({
                        "file_path": file_path,
                        "chunk": chunk_text,
                        "chunk_id": chunk_id,
                    })
                    chunk_id += 1

                if is_last_line:
                    break
                overlap_lines: list[str] = []
                overlap_len = 0
                for back_line in reversed(current_lines):
                    overlap_lines.insert(0, back_line)
                    overlap_len += len(back_line)
                    if overlap_len >= overlap:
                        break

                current_lines = overlap_lines
                current_len = overlap_len

    print(f"Created {len(all_chunks)} chunks from {len(files_data)} files.")
    return all_chunks

In [52]:
def create_embeddings(chunks: list[dict], model: SentenceTransformer) -> np.ndarray:
    """
    Generate normalized embeddings for a list of code chunks.

    Batching all chunks through model.encode() (instead of one at a time) is
    much faster since it lets the model use vectorized operations. Normalizing
    embeddings lets us use a cheap inner product in FAISS as an exact stand-in
    for cosine similarity.

    Args:
        chunks: Output of chunk_code() -> [{"file_path", "chunk", "chunk_id"}, ...]
        model: A loaded SentenceTransformer model.

    Returns:
        A (num_chunks, embedding_dim) float32 numpy array of unit-normalized embeddings.
    """
    texts = [c["chunk"] for c in chunks]
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return embeddings.astype("float32")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: all-MiniLM-L6-v2


In [53]:
def build_faiss(chunks: list[dict], model: SentenceTransformer,
                 index_dir: str = INDEX_DIR):
    """
    Build a fresh FAISS index from repository chunks and persist it to disk.

    Uses IndexFlatIP (inner product) which, on normalized vectors, is
    mathematically equivalent to cosine similarity and gives exact (not
    approximate) results — the right tradeoff at single-repo scale.

    Args:
        chunks: Output of chunk_code().
        model: Loaded SentenceTransformer used to embed the chunks.
        index_dir: Folder to save the FAISS index + metadata into.

    Returns:
        (faiss_index, metadata) where metadata[i] describes vector i.
    """
    os.makedirs(index_dir, exist_ok=True)

    embeddings = create_embeddings(chunks, model)
    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, os.path.join(index_dir, "index.faiss"))
    with open(os.path.join(index_dir, "metadata.pkl"), "wb") as f:
        pickle.dump(chunks, f)

    print(f"Built and saved FAISS index with {index.ntotal} vectors.")
    return index, chunks

def load_faiss(index_dir: str = INDEX_DIR):
    """
    Load a previously saved FAISS index + metadata, if present.

    Returns:
        (faiss_index, metadata) if found, otherwise (None, None).
    """
    index_path = os.path.join(index_dir, "index.faiss")
    metadata_path = os.path.join(index_dir, "metadata.pkl")

    if not (os.path.exists(index_path) and os.path.exists(metadata_path)):
        return None, None

    index = faiss.read_index(index_path)
    with open(metadata_path, "rb") as f:
        metadata = pickle.load(f)

    print(f"Loaded existing FAISS index with {index.ntotal} vectors.")
    return index, metadata


In [54]:
def build_or_load_repository(repo_url: str, embedding_model: SentenceTransformer,
                              force_rebuild: bool = False):
    """
    Full ingestion pipeline: clone -> read -> chunk -> embed -> FAISS index.
    Reuses a cached index if one already exists on disk (unless force_rebuild=True).

    Args:
        repo_url: HTTPS URL of a public GitHub repository.
        embedding_model: Loaded SentenceTransformer to use for embedding.
        force_rebuild: If True, ignores any cached index and rebuilds from scratch.

    Returns:
        (faiss_index, metadata)
    """
    if not force_rebuild:
        index, metadata = load_faiss()
        if index is not None:
            return index, metadata

    repo_path = clone_repo(repo_url)
    files_data = read_repository(repo_path)

    if not files_data:
        raise RuntimeError("No supported files found in this repository.")

    chunks = chunk_code(files_data)
    index, metadata = build_faiss(chunks, embedding_model)
    return index, metadata

In [55]:
def search_repository(query: str, index, metadata: list[dict],
                       embedding_model: SentenceTransformer,
                       k: int = TOP_K) -> list[dict]:
    """
    Retrieve the top-k most relevant chunks for a natural language query.

    Args:
        query: The user's question.
        index: FAISS index built over the repository's chunk embeddings.
        metadata: Parallel list describing each vector in the index.
        embedding_model: The same SentenceTransformer used to build the index.
        k: Number of chunks to retrieve.

    Returns:
        List of dicts: {"file_path", "chunk", "score"}, sorted by relevance (desc).
    """
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        entry = metadata[idx]
        results.append({
            "file_path": entry["file_path"],
            "chunk": entry["chunk"],
            "score": float(score),
        })

    return results

In [56]:
def _format_context(results: list[dict]) -> str:
    """Turn retrieved chunks into a labeled context block for the prompt."""
    blocks = []
    for r in results:
        blocks.append(
            f"### File: {r['file_path']} (similarity: {r['score']:.3f})\n"
            f"```\n{r['chunk']}\n```"
        )
    return "\n\n".join(blocks)

In [57]:
def ask_repository(query: str, index, metadata: list[dict],
                    embedding_model: SentenceTransformer, llm: ChatGroq,
                    k: int = TOP_K) -> str:
    """
    Answer a natural-language question about the repository using RAG.

    Args:
        query: The user's question.
        index: FAISS index for the repository.
        metadata: Chunk metadata parallel to the index.
        embedding_model: SentenceTransformer used for retrieval.
        llm: Configured ChatGroq client.
        k: Number of chunks to retrieve as context.

    Returns:
        The LLM's answer as a string.
    """
    results = search_repository(query, index, metadata, embedding_model, k=k)

    if not results:
        return "I couldn't find enough information inside this repository."

    context = _format_context(results)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Repository context:\n\n{context}\n\nQuestion: {query}"),
    ]

    response = llm.invoke(messages)
    return response.content

In [58]:
def summarize_repository(index, metadata, embedding_model, llm) -> str:
    """High-level summary of what the repository does and how it's structured."""
    query = (
        "Explain what this repository does overall: its purpose, main components, "
        "and high-level architecture."
    )
    return ask_repository(query, index, metadata, embedding_model, llm, k=8)

def explain_file(file_path: str, metadata: list[dict], llm) -> str:
    """
    Explain a specific file by gathering ALL of its chunks (not similarity
    search, since we want the whole file, not just parts similar to a query).
    """
    file_chunks = [m for m in metadata if m["file_path"] == file_path]

    if not file_chunks:
        return f"I couldn't find a file named '{file_path}' in this repository."

    context = "\n\n".join(
        f"```\n{c['chunk']}\n```" for c in sorted(file_chunks, key=lambda c: c["chunk_id"])
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Repository context (full contents of '{file_path}'):\n\n{context}\n\n"
            f"Question: Explain what this file does, its key functions/classes, "
            f"and how it fits into the project."
        )),
    ]
    return llm.invoke(messages).content

def find_relevant_files(query: str, index, metadata: list[dict],
                         embedding_model, k: int = TOP_K) -> list[str]:
    """Return just the distinct filenames most relevant to a query (no LLM call needed)."""
    results = search_repository(query, index, metadata, embedding_model, k=k)
    seen = []
    for r in results:
        if r["file_path"] not in seen:
            seen.append(r["file_path"])
    return seen

def summarize_readme(metadata: list[dict], llm) -> str:
    """Summarize README.md specifically, if present."""
    readme_chunks = [m for m in metadata if m["file_path"].lower().endswith("readme.md")]

    if not readme_chunks:
        return "This repository doesn't appear to contain a README.md."

    context = "\n\n".join(
        c["chunk"] for c in sorted(readme_chunks, key=lambda c: c["chunk_id"])
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"README content:\n\n{context}\n\nQuestion: Summarize this README."),
    ]
    return llm.invoke(messages).content

def folder_structure_overview(llm, repo_path: str = REPO_DIR) -> str:
    """
    Build a textual tree of top-level (and one level deep) folders, then ask
    the LLM to explain what each major folder is likely responsible for.
    """
    tree_lines = []
    for root, dirnames, filenames in os.walk(repo_path):
        dirnames[:] = [d for d in dirnames if d not in IGNORED_DIRS]
        depth = root[len(repo_path):].count(os.sep)
        if depth > 2:
            dirnames[:] = []
            continue
        indent = "  " * depth
        tree_lines.append(f"{indent}{os.path.basename(root) or '.'}/")
        for filename in filenames[:10]:
            tree_lines.append(f"{indent}  {filename}")

    tree_text = "\n".join(tree_lines)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Repository folder structure:\n\n{tree_text}\n\n"
            f"Question: Explain what each major top-level folder is likely responsible for."
        )),
    ]
    return llm.invoke(messages).content


In [59]:
index = None
metadata = None
embedding_model = None
llm = None

In [60]:
@tool
def search_repository_tool(query: str) -> str:
    """Semantic search over the repository's code and docs. Input: a natural
    language query (e.g. 'JWT token validation'). Returns the most relevant
    code snippets with their file paths and similarity scores. Use this
    FIRST when you don't yet know which file(s) are relevant."""
    results = search_repository(query, index, metadata, embedding_model, k=TOP_K)
    if not results:
        return "No relevant chunks found for this query."
    return "\n\n".join(
        f"File: {r['file_path']} (score: {r['score']:.3f})\n{r['chunk'][:600]}"
        for r in results
    )

In [61]:
@tool
def find_relevant_files_tool(query: str) -> str:
    """Returns just a list of filenames most relevant to a query, with no
    explanation. Input: a natural language query. Use this when the user
    asks 'which file(s)' or 'where' something lives and only wants
    filenames back, or to quickly identify candidates before explain_file."""
    files = find_relevant_files(query, index, metadata, embedding_model, k=TOP_K)
    return "\n".join(files) if files else "No relevant files found for this query."

In [62]:
@tool
def explain_file_tool(file_path: str) -> str:
    """Explains ONE specific file in detail: its purpose, key functions/
    classes, and how it fits into the project. Input: the EXACT relative
    file path (e.g. 'app/auth.py'), not a guess. Use this AFTER you've
    identified the right file via search_repository or find_relevant_files."""
    return explain_file(file_path.strip(), metadata, llm)

In [63]:
@tool
def summarize_repository_tool(_: str = "") -> str:
    """Produces a high-level summary of the ENTIRE repository's purpose,
    main components, and architecture. Input is ignored. Use this when the
    user asks for an overview of the whole project, not one feature/file."""
    return summarize_repository(index, metadata, embedding_model, llm)

In [64]:
@tool
def summarize_readme_tool(_: str = "") -> str:
    """Summarizes the repository's README.md specifically. Input is
    ignored. Use this when the user asks about the README or wants the
    project's own description of itself."""
    return summarize_readme(metadata, llm)

In [65]:
@tool
def folder_structure_overview_tool(_: str = "") -> str:
    """Explains what each major top-level folder in the repository is
    responsible for. Input is ignored. Use this when the user asks about
    the project's folder/directory structure or codebase organization."""
    return folder_structure_overview(llm)

In [66]:
AGENT_SYSTEM_PROMPT = """You are an expert software engineer agent with access to several \
repository analysis tools (search_repository_tool, find_relevant_files_tool, \
explain_file_tool, summarize_repository_tool, summarize_readme_tool, \
folder_structure_overview_tool).

Your job: answer the user's question about the repository by reasoning step by step \
and deciding for yourself which tool(s) to call, in what order, and how many times.

Rules you MUST follow:
1. NEVER answer from general/outside knowledge about the framework or library. \
Only answer using information you obtained from your tools.
2. Use multiple tools if a question requires it. A typical flow for a specific \
feature question (e.g. "explain authentication") is:
   search_repository_tool -> identify the relevant file(s) -> explain_file_tool -> \
   (optionally) summarize_repository_tool for broader context -> final answer.
3. For broad questions ("what does this repo do") prefer summarize_repository_tool or \
folder_structure_overview_tool directly.
4. If a tool's observation doesn't contain what you need, try a different tool or a \
reformulated query before giving up.
5. If, after using your tools, the answer truly cannot be found, say exactly: \
"I couldn't find enough information inside this repository."
6. Always mention the specific filename(s) your final answer is based on.
7. Never hallucinate function names, file names, or behavior that wasn't returned by \
a tool observation.
8. Be clear and structured in your final answer. Use bullet points where helpful.

When using explain_file_tool:

ONLY use filenames that were returned by
find_relevant_files_tool
or
search_repository_tool.

Never invent file paths.

If no suitable filename exists,
do not call explain_file_tool.
"""

In [67]:
def main():
    global index, metadata, embedding_model, llm

    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

    print("Loading embedding model...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0,
        api_key=os.environ["GROQ_API_KEY"],
    )

    repo_url = input("Enter a public GitHub repository URL: ").strip()
    index, metadata = build_or_load_repository(repo_url, embedding_model)

    # --- Existing functions, now available as @tool functions above ---
    tools = [
        search_repository_tool,
        find_relevant_files_tool,
        explain_file_tool,
        summarize_repository_tool,
        summarize_readme_tool,
        folder_structure_overview_tool,
    ]

    # --- Agent: reason -> pick tool(s) -> observe -> repeat -> final answer ---
    agent = create_react_agent(llm, tools, prompt=AGENT_SYSTEM_PROMPT)

    print("\nRepository indexed. Ask ONE natural-language question at a time --")
    print("the agent will decide on its own which tool(s) to use and in what order.")
    print("Examples: 'Explain authentication', 'What does this repo do?', "
          "'Which files handle routing?'")
    print("Type 'exit' to quit.\n")

    while True:
        query = input("Ask> ").strip()
        if not query:
            continue
        if query.lower() in {"exit", "quit"}:
            break

        # No manual routing anymore -- the agent reasons, picks tools, and
        # loops (Thought -> Action -> Observation) until it has a final answer.
        result = agent.invoke({"messages": [HumanMessage(content=query)]})

        for msg in result["messages"]:
            if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
                for call in msg.tool_calls:
                    print(f"[Thought -> Action] {call['name']}({call['args']})")
            elif isinstance(msg, ToolMessage):
                print(f"[Observation from '{msg.name}']: {str(msg.content)[:300]}\n")

        print(f"\nFinal Answer: {result['messages'][-1].content}\n")


In [68]:
if __name__ == "__main__":
    main()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Enter a public GitHub repository URL: https://github.com/avnigoel54-rgb/Real-Time-Snapchat-Style-Filters


/tmp/ipykernel_8071/4241353054.py:30: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=AGENT_SYSTEM_PROMPT)


Loaded existing FAISS index with 9 vectors.

Repository indexed. Ask ONE natural-language question at a time --
the agent will decide on its own which tool(s) to use and in what order.
Examples: 'Explain authentication', 'What does this repo do?', 'Which files handle routing?'
Type 'exit' to quit.

Ask> explain the content of this repository.
[Thought -> Action] summarize_repository_tool({})
[Observation from 'summarize_repository_tool']: Based on the provided repository context, I can explain the overall purpose, main components, and high-level architecture of this project.

**Purpose:**
This repository is a Computer Vision project that implements real-time Snapchat-like filters on a live webcam feed using Python and OpenCV. The pro


Final Answer: The final answer is based on the following filenames:

* snapchat.py

Ask> exit
